# Binance Backtest: ATR-Based Stop/TP with Trailing (Python)

This notebook fetches historical OHLCV data from the **Binance** public REST API and runs a bar-by-bar backtest that mirrors a typical TradingView Pine strategy using:

- **Entries**: Simple, configurable signals (default: SMA crossover).  
- **Exits**: Fixed Stop-Loss and Take-Profit, plus optional **trailing stop** with 
  - `trail_offset` (activation distance)
  - `trail_points` (distance that the stop trails behind favorable price)
- **ATR**-based distances in **price units** (not ticks).

> **Note:** TradingView’s `strategy.exit()` expects *price units* for `trail_points` and `trail_offset`. This notebook follows that convention and latches ATR-based distances at the time of signal.

**Last generated:** 2026-03-13 20:37 UTC

In [ ]:
# %% [markdown]
# ## 1) Imports & Global Settings

import math
import time
import json
from dataclasses import dataclass
from typing import List, Optional, Tuple
import datetime as dt

import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.width', 160)
pd.set_option('display.max_columns', 50)

# Plot style
plt.style.use('seaborn-v0_8')

## 2) Binance Data Fetching

This section fetches historical klines from Binance's public endpoint `GET /api/v3/klines`.

- No API key is required for public klines.  
- Each request returns up to **1000** candles. We paginate until the requested time range is complete.  
- Time parameters are in **milliseconds** since epoch.

**Parameters you'll set below:**
- `symbol` (e.g., `BTCUSDT`)
- `interval` (e.g., `1h`, `15m`, `1d`)
- `start` / `end` datetimes

In [ ]:
# %%
BINANCE_BASE = 'https://api.binance.com'

INTERVALS = {
    '1m':'1m','3m':'3m','5m':'5m','15m':'15m','30m':'30m',
    '1h':'1h','2h':'2h','4h':'4h','6h':'6h','8h':'8h','12h':'12h',
    '1d':'1d','3d':'3d','1w':'1w','1M':'1M'
}

def to_millis(ts: dt.datetime) -> int:
    if ts.tzinfo is None:
        # treat as UTC if naive
        ts = ts.replace(tzinfo=dt.timezone.utc)
    return int(ts.timestamp() * 1000)

def fetch_klines(symbol: str, interval: str, start_time: dt.datetime, end_time: dt.datetime, limit: int = 1000, pause: float = 0.3) -> pd.DataFrame:
    """Fetch historical klines from Binance.

    Parameters
    ----------
    symbol : str
        e.g., 'BTCUSDT'
    interval : str
        e.g., '1h', '15m', '1d'
    start_time : datetime
    end_time : datetime
    limit : int
        Max candles per request (Binance caps at 1000).
    pause : float
        Sleep between requests to be gentle on API.
    """
    assert interval in INTERVALS.values(), f'Unsupported interval: {interval}'

    start_ms = to_millis(start_time)
    end_ms = to_millis(end_time)
    url = f'{BINANCE_BASE}/api/v3/klines'

    all_rows = []
    cur = start_ms
    while cur < end_ms:
        params = {
            'symbol': symbol.upper(),
            'interval': interval,
            'startTime': cur,
            'endTime': end_ms,
            'limit': limit,
        }
        resp = requests.get(url, params=params, timeout=30)
        resp.raise_for_status()
        batch = resp.json()
        if not batch:
            break

        all_rows.extend(batch)
        # Advance: Binance returns klines with open time ascending; use last open time + interval step
        last_open = batch[-1][0]
        if len(batch) < limit:
            break

        # Move start beyond last returned open time
        cur = last_open + 1
        time.sleep(pause)

    if not all_rows:
        return pd.DataFrame(columns=['open_time','open','high','low','close','volume','close_time','trades'])

    # Per Binance docs, fields are:
    # 0 open time(ms), 1 open, 2 high, 3 low, 4 close, 5 volume, 6 close time(ms), 7 quote asset volume,
    # 8 number of trades, 9 taker buy base asset volume, 10 taker buy quote asset volume, 11 ignore
    cols = ['open_time','open','high','low','close','volume','close_time','quote_asset_volume','trades','taker_buy_base','taker_buy_quote','ignore']
    df = pd.DataFrame(all_rows, columns=cols)
    for c in ['open','high','low','close','volume','quote_asset_volume','taker_buy_base','taker_buy_quote']:
        df[c] = pd.to_numeric(df[c], errors='coerce')
    df['open_time'] = pd.to_datetime(df['open_time'], unit='ms', utc=True)
    df['close_time'] = pd.to_datetime(df['close_time'], unit='ms', utc=True)
    df = df.drop(columns=['quote_asset_volume','taker_buy_base','taker_buy_quote','ignore'])
    df = df.rename(columns={'trades':'num_trades'})
    df = df.set_index('open_time').sort_index()
    return df

## 3) Indicators: ATR and Example Signals

- **ATR** is computed in *price units* and will be used to derive stop/TP/trailing distances.
- Example **signals**: SMA crossover (you can replace with your own conditions).

In [ ]:
# %%
def compute_atr(df: pd.DataFrame, length: int = 14) -> pd.Series:
    high = df['high']
    low = df['low']
    close = df['close']
    prev_close = close.shift(1)
    tr = pd.concat([
        high - low,
        (high - prev_close).abs(),
        (low - prev_close).abs()
    ], axis=1).max(axis=1)
    atr = tr.rolling(length, min_periods=length).mean()
    return atr

def compute_sma(series: pd.Series, length: int) -> pd.Series:
    return series.rolling(length, min_periods=length).mean()

def example_signals(df: pd.DataFrame, fast: int = 20, slow: int = 50) -> Tuple[pd.Series, pd.Series]:
    sma_fast = compute_sma(df['close'], fast)
    sma_slow = compute_sma(df['close'], slow)
    long_sig = (sma_fast > sma_slow) & (sma_fast.shift(1) <= sma_slow.shift(1))
    short_sig = (sma_fast < sma_slow) & (sma_fast.shift(1) >= sma_slow.shift(1))
    return long_sig.fillna(False), short_sig.fillna(False)

## 4) Backtest Engine (bar-by-bar)

This engine:
- Enters at **next bar open** when a signal occurs at bar close.
- Latches ATR-based distances at the **signal bar** and applies them at entry.
- Manages **stop**, **take profit**, and **trailing** with:
  - `trail_offset` → activation distance (price must move this much in favor before trailing turns on).
  - `trail_points` → trailing gap from the best favorable price after activation.
- Checks exits within a bar using OHLC:
  - For **longs**: if `low <= stop` then stop first; else if `high >= tp` then TP.
  - For **shorts**: if `high >= stop` then stop first; else if `low <= tp` then TP.

> You can refine these rules (e.g., intrabar path assumptions, fees/slippage) to better match TradingView.

In [ ]:
# %%
from dataclasses import dataclass

@dataclass
class Bar:
    time: pd.Timestamp
    open: float
    high: float
    low: float
    close: float

@dataclass
class Trade:
    side: str                  # 'long' or 'short'
    entry_time: pd.Timestamp
    entry_price: float
    exit_time: Optional[pd.Timestamp] = None
    exit_price: float = math.nan
    exit_reason: str = ''

class BacktestEngine:
    def __init__(self):
        self.pos_side: Optional[str] = None
        self.pos_entry_price: Optional[float] = None
        self.pos_highest: Optional[float] = None  # for long
        self.pos_lowest: Optional[float] = None   # for short
        self.trailing_active: bool = False
        self.stop_level: Optional[float] = None
        self.tp_level: Optional[float] = None
        self.trades: List[Trade] = []
        # pending entry = (side, signal_time, stop_dist, tp_dist, trail_points, trail_offset)
        self.pending_entry: Optional[Tuple[str, pd.Timestamp, float, float, Optional[float], Optional[float]]] = None

    def _reset_position_state(self):
        self.pos_side = None
        self.pos_entry_price = None
        self.pos_highest = None
        self.pos_lowest = None
        self.trailing_active = False
        self.stop_level = None
        self.tp_level = None
        self.pending_entry = None

    def on_bar(self, bar: Bar,
               long_signal: bool,
               short_signal: bool,
               stop_dist: float,
               tp_dist: float,
               trail_points: Optional[float],
               trail_offset: Optional[float]):
        """Process one bar.
        stop_dist/tp_dist/trailing params are *price distances* computed for the current bar; if a signal fires,
        we *latch* these distances for use at the next-bar entry.
        """
        # 1) Execute pending entries at this bar's open
        if self.pending_entry is not None and self.pos_side is None:
            side, sig_time, latched_stop, latched_tp, latched_points, latched_offset = self.pending_entry
            entry_price = bar.open
            self.pos_side = side
            self.pos_entry_price = entry_price
            if side == 'long':
                self.stop_level = entry_price - latched_stop
                self.tp_level = entry_price + latched_tp
                self.pos_highest = entry_price
            else:
                self.stop_level = entry_price + latched_stop
                self.tp_level = entry_price - latched_tp
                self.pos_lowest = entry_price
            self.trailing_active = False
            self.trades.append(Trade(side=side, entry_time=sig_time, entry_price=entry_price))
            self.pending_entry = None

        # 2) Manage open position (update trailing and check exits on this bar)
        if self.pos_side is not None:
            if self.pos_side == 'long':
                # update highest favorable price
                self.pos_highest = max(self.pos_highest, bar.high)
                # activate trailing if applicable
                if (trail_points is not None and trail_offset is not None) and not self.trailing_active:
                    if (self.pos_highest - self.pos_entry_price) >= trail_offset:
                        self.trailing_active = True
                # update trailing stop position (ratchet only tighter)
                if (trail_points is not None and self.trailing_active):
                    self.stop_level = max(self.stop_level, self.pos_highest - trail_points)

                exit_reason, exit_price = None, None
                if bar.low <= self.stop_level:
                    exit_reason = 'trail_stop' if self.trailing_active else 'stop'
                    exit_price = self.stop_level
                elif bar.high >= self.tp_level:
                    exit_reason = 'tp'
                    exit_price = self.tp_level

            else:  # short
                self.pos_lowest = min(self.pos_lowest, bar.low)
                if (trail_points is not None and trail_offset is not None) and not self.trailing_active:
                    if (self.pos_entry_price - self.pos_lowest) >= trail_offset:
                        self.trailing_active = True
                if (trail_points is not None and self.trailing_active):
                    self.stop_level = min(self.stop_level, self.pos_lowest + trail_points)

                exit_reason, exit_price = None, None
                if bar.high >= self.stop_level:
                    exit_reason = 'trail_stop' if self.trailing_active else 'stop'
                    exit_price = self.stop_level
                elif bar.low <= self.tp_level:
                    exit_reason = 'tp'
                    exit_price = self.tp_level

            if exit_reason is not None:
                # Close the last open trade
                for i in range(len(self.trades)-1, -1, -1):
                    if math.isnan(self.trades[i].exit_price):
                        self.trades[i].exit_price = exit_price
                        self.trades[i].exit_time = bar.time
                        self.trades[i].exit_reason = exit_reason
                        break
                self._reset_position_state()

        # 3) If flat, review signals at close and schedule entry for next bar open
        if self.pos_side is None:
            # If both signals occur, prefer long (customize as needed)
            if long_signal and not short_signal:
                self.pending_entry = ('long', bar.time, stop_dist, tp_dist, trail_points, trail_offset)
            elif short_signal and not long_signal:
                self.pending_entry = ('short', bar.time, stop_dist, tp_dist, trail_points, trail_offset)
            # elif long_signal and short_signal:
            #     self.pending_entry = ('long', bar.time, stop_dist, tp_dist, trail_points, trail_offset)

## 5) Configure, Run, and Evaluate

Set your parameters (symbol, interval, date range, ATR length, and ATR multiples for stop/TP/trailing). Then run the backtest, review the trade list, performance metrics, and plots.

In [ ]:
# %%
# --- User Parameters ---
symbol = 'BTCUSDT'
interval = '1h'  # choices include: '15m','1h','4h','1d', etc.
start = dt.datetime(2023, 1, 1, tzinfo=dt.timezone.utc)
end   = dt.datetime(2024, 1, 1, tzinfo=dt.timezone.utc)

atr_len = 14
stop_mult = 2.0     # SL = stop_mult * ATR
tp_mult = 3.0       # TP = tp_mult * ATR
use_trailing = True
trail_mult = 1.2    # trail_points = trail_mult * ATR; trail_offset = trail_mult * ATR

# Example signal params
sma_fast = 20
sma_slow = 50

print(f'Fetching {symbol} {interval} from {start} to {end} ...')
df = fetch_klines(symbol, interval, start, end)
if df.empty:
    raise SystemExit('No data returned. Check symbol/interval/date range.')

# Compute indicators
df['atr'] = compute_atr(df, atr_len)
df['sma_fast'] = compute_sma(df['close'], sma_fast)
df['sma_slow'] = compute_sma(df['close'], sma_slow)

# Signals (at bar close)
long_sig, short_sig = example_signals(df, sma_fast, sma_slow)

# Distances in price units *at each bar* (latched when signal occurs)
df['stop_dist'] = stop_mult * df['atr']
df['tp_dist']   = tp_mult * df['atr']
if use_trailing:
    df['trail_points'] = trail_mult * df['atr']
    df['trail_offset'] = trail_mult * df['atr']
else:
    df['trail_points'] = np.nan
    df['trail_offset'] = np.nan

# Build bar objects
bars = [Bar(time=ts, open=row['open'], high=row['high'], low=row['low'], close=row['close'])
        for ts, row in df.iterrows()]

engine = BacktestEngine()

# Iterate through bars
for i, bar in enumerate(bars):
    engine.on_bar(
        bar=bar,
        long_signal=bool(long_sig.iloc[i]),
        short_signal=bool(short_sig.iloc[i]),
        stop_dist=float(df['stop_dist'].iloc[i]) if not math.isnan(df['stop_dist'].iloc[i]) else np.nan,
        tp_dist=float(df['tp_dist'].iloc[i]) if not math.isnan(df['tp_dist'].iloc[i]) else np.nan,
        trail_points=(float(df['trail_points'].iloc[i]) if not math.isnan(df['trail_points'].iloc[i]) else None),
        trail_offset=(float(df['trail_offset'].iloc[i]) if not math.isnan(df['trail_offset'].iloc[i]) else None),
    )

# Convert trade log to DataFrame
trades = pd.DataFrame([{
    'side': t.side,
    'entry_time': t.entry_time,
    'entry_price': t.entry_price,
    'exit_time': t.exit_time,
    'exit_price': t.exit_price,
    'exit_reason': t.exit_reason,
} for t in engine.trades])

# Drop open trades without exit
trades = trades.dropna(subset=['exit_price']).copy()
trades['pnl_pts'] = np.where(trades['side']=='long',
                             trades['exit_price'] - trades['entry_price'],
                             trades['entry_price'] - trades['exit_price'])

# Assume 1 unit per trade; equity curve starts at 0 and accumulates P&L in price units.
trades['cum_pnl_pts'] = trades['pnl_pts'].cumsum()

# Basic stats
n_trades = len(trades)
wins = (trades['pnl_pts'] > 0).sum()
win_rate = wins / n_trades * 100 if n_trades else 0.0
gross_profit = trades.loc[trades['pnl_pts'] > 0, 'pnl_pts'].sum()
gross_loss = -trades.loc[trades['pnl_pts'] <= 0, 'pnl_pts'].sum()
profit_factor = (gross_profit / gross_loss) if gross_loss > 0 else np.inf

print(f'Trades: {n_trades}, Win%: {win_rate:.2f}%, Profit Factor: {profit_factor:.2f}')

trades.tail()

## 6) Plots: Equity Curve & Drawdown

In [ ]:
# %%
if not trades.empty:
    # Build equity series indexed by exit_time
    eq = trades.set_index('exit_time')['pnl_pts'].cumsum()
    eq = eq.asfreq('D').fillna(method='ffill')  # resample for smoother plot spacing
    dd = eq - eq.cummax()

    fig, ax = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
    ax[0].plot(eq.index, eq.values, label='Equity (points)')
    ax[0].set_title('Equity Curve (points)')
    ax[0].legend()
    ax[0].grid(True, alpha=0.3)

    ax[1].plot(dd.index, dd.values, color='crimson', label='Drawdown (points)')
    ax[1].set_title('Drawdown (points)')
    ax[1].legend()
    ax[1].grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print('No closed trades to plot yet.')

## 7) Export Trades

In [ ]:
# %%
out_csv = f'trades_{symbol}_{interval}.csv'
trades.to_csv(out_csv, index=False)
print(f'Saved trades to: {out_csv}')
trades.head()

## 8) Notes & Parity with TradingView

- **Distances in price units**: `stop_dist`, `tp_dist`, `trail_points`, and `trail_offset` are in *price units*, aligned with TradingView's `strategy.exit()` parameter expectations.
- **Trailing semantics**:
  - `trail_offset` = profit required to **activate** the trailing stop.
  - `trail_points` = **gap** behind the best favorable price once activated.
- **Entry/Exit timing**: Signals evaluated at bar close; entries filled at **next bar open**. Exits are checked against the bar's OHLC with a simple priority rule (stop before TP when both would hit in the same bar for longs, and vice-versa for shorts). Adjust as needed to match your Pine setup.
- **Slippage / fees**: Not modeled by default. You can subtract fees per trade or per side and add slippage to entry/exit prices.
- **Sizing**: The example assumes 1 unit per trade. You can add position sizing logic (risk-based sizing from stop distance, account equity, etc.).
- **Signals**: SMA crossover is a placeholder. Replace with your own conditions to mirror your Pine strategy.

### Common Extensions
- Add commissions and slippage parameters.
- Resolve same-bar SL vs TP with a chosen **intrabar path** assumption (e.g., O-H-L-C or O-L-H-C) or use a lower timeframe.
- Export detailed logs (per-bar state, equity, run diagnostics).
- Compare results vs. TradingView exports for validation.